# Baseline de prédiction du rendement — `/predict`

**Objectif** : poser une référence simple pour le service `/predict`.

| Modèle | Rôle |
|---|---|
| `DummyRegressor` | référence naïve : prédit toujours le rendement moyen |
| `LinearRegression` | première baseline |

**Métriques**

- **RMSE** : erreur quadratique moyenne, en t/ha ; elle pénalise davantage les grosses erreurs.
- **R²** : part de la variance du rendement expliquée par le modèle ; 0 pour une prédiction constante, 1 pour une
  prédiction parfaite.
- **MAE** : erreur absolue moyenne, en t/ha.

## Imports

In [1]:
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

from agritech.config import SEED
from agritech.evaluation import cross_validate_regressor, format_cv_metrics
from agritech.modeling import RunInfo, log_model_run
from agritech.notification import notify
from agritech.predict_config import (
    PREDICT_CATEGORICAL,
    PREDICT_CV_FOLDS,
    PREDICT_CV_SHUFFLE,
    PREDICT_DATASET,
    PREDICT_FEATURES,
    PREDICT_FEATURES_DROP,
    PREDICT_NUMERIC,
    PREDICT_ROWS,
    PREDICT_TARGET,
    PREDICT_TEST_SIZE,
)
from agritech.preprocessing import build_pipeline, count_encoded_columns
from agritech.tracking import log_run, run_tags, setup_mlflow
from agritech.training_data import feature_types, kfold_cv, load_dataset, protocol_params, split

# Données

## Lecture et contrôles de base

Dataset produit par le notebook 06. Les 231 rendements négatifs en ont été retirés.

In [2]:
df = load_dataset(PREDICT_DATASET, PREDICT_ROWS, PREDICT_FEATURES + [PREDICT_TARGET], non_negative=[PREDICT_TARGET])

print("\ntypes :")
print(df.dtypes.to_string())

fichier            : data/processed/predict_training_dataset.csv
lignes x colonnes  : (999769, 10)
valeurs manquantes : 0
valeurs < 0        : 0 (Yield_tons_per_hectare)

types :
Crop                       object
Soil_Type                  object
Rainfall_mm               float64
Temperature_Celsius       float64
Fertilizer_Used              bool
Irrigation_Used              bool
Region                     object
Weather_Condition          object
Days_to_Harvest             int64
Yield_tons_per_hectare    float64


## Variables candidates

- **Catégorielles** : `Crop`, `Soil_Type`, `Region`, `Weather_Condition`, `Fertilizer_Used`, `Irrigation_Used`.
- **Numériques** : `Rainfall_mm`, `Temperature_Celsius`, `Days_to_Harvest`.

Deux jeux sont comparés plus bas : **toutes les variables**, soit les 9 candidates, et les **variables réduites**,
sans `Region`, `Weather_Condition` ni `Days_to_Harvest` (`PREDICT_FEATURES_DROP`).

In [3]:
VARIABLES_REDUITES = [variable for variable in PREDICT_FEATURES if variable not in PREDICT_FEATURES_DROP]

print("catégorielles :", PREDICT_CATEGORICAL)
print("numériques    :", PREDICT_NUMERIC)
print("retirées      :", PREDICT_FEATURES_DROP)
print("réduites      :", VARIABLES_REDUITES)
print("\nmodalités par variable catégorielle :")
print(df[PREDICT_CATEGORICAL].nunique().to_string())

catégorielles : ['Crop', 'Soil_Type', 'Fertilizer_Used', 'Irrigation_Used', 'Region', 'Weather_Condition']
numériques    : ['Rainfall_mm', 'Temperature_Celsius', 'Days_to_Harvest']
retirées      : ['Region', 'Weather_Condition', 'Days_to_Harvest']
réduites      : ['Crop', 'Soil_Type', 'Fertilizer_Used', 'Irrigation_Used', 'Rainfall_mm', 'Temperature_Celsius']

modalités par variable catégorielle :
Crop                 6
Soil_Type            6
Fertilizer_Used      2
Irrigation_Used      2
Region               4
Weather_Condition    3


# Découpage train / test

80 % pour l'entraînement, 20 % pour le test, de façon aléatoire et reproductible (`random_state=42`), avant tout
preprocessing appris.

**Le jeu de test est réservé à l'évaluation finale du modèle retenu.**

In [4]:
X_train, X_test, y_train, y_test = split(df, PREDICT_FEATURES, PREDICT_TARGET, PREDICT_TEST_SIZE, SEED)

total   : 999769 lignes
X_train : (799815, 9)
X_test  : (199954, 9), réservé à l'évaluation finale
y_train : (799815,)
y_test  : (199954,)


# Protocole de validation croisée

Les modèles sont comparés par validation croisée à 5 folds, sur le jeu d'entraînement uniquement
(`shuffle=True`, `random_state=42`). Les folds sont les mêmes pour tous les modèles : les scores sont directement
comparables. Ce protocole servira aux modèles `/predict` suivants.

In [5]:
cv = kfold_cv(PREDICT_CV_FOLDS, PREDICT_CV_SHUFFLE, SEED)
folds = [(len(apprentissage), len(evaluation)) for apprentissage, evaluation in cv.split(X_train)]

for numero, (n_apprentissage, n_evaluation) in enumerate(folds, start=1):
    print(f"fold {numero} : {n_apprentissage} lignes d'apprentissage | {n_evaluation} d'évaluation")

validation croisée : KFold(n_splits=5, random_state=42, shuffle=True)
fold 1 : 639852 lignes d'apprentissage | 159963 d'évaluation
fold 2 : 639852 lignes d'apprentissage | 159963 d'évaluation
fold 3 : 639852 lignes d'apprentissage | 159963 d'évaluation
fold 4 : 639852 lignes d'apprentissage | 159963 d'évaluation
fold 5 : 639852 lignes d'apprentissage | 159963 d'évaluation


In [6]:
# mêmes tags et mêmes paramètres de protocole pour tous les runs /predict
experience = setup_mlflow("predict")
TAGS = run_tags(service="predict", stage="baseline", notebook="07_predict_training_baseline.ipynb")
PARAMS_PROTOCOLE = protocol_params(PREDICT_DATASET, X_train, X_test, PREDICT_TEST_SIZE, SEED, PREDICT_CV_FOLDS, PREDICT_CV_SHUFFLE)

expérience MLflow : oc_p12_agritech_predict


# Référence naïve : `DummyRegressor`

`DummyRegressor(strategy="mean")` prédit toujours le rendement moyen. Il fixe le niveau minimal à battre.

In [7]:
dummy = DummyRegressor(strategy="mean")
cv_dummy = cross_validate_regressor(dummy, X_train, y_train, cv)

print(f"écart-type du rendement dans le train : {y_train.std(ddof=0):.3f} t/ha")
print(format_cv_metrics(cv_dummy))

écart-type du rendement dans le train : 1.695 t/ha
RMSE 1.6952 ± 0.0027 t/ha | MAE 1.3884 ± 0.0031 t/ha | R² -0.0000 ± 0.0000


In [8]:
log_run(
    "dummy_baseline_cv",
    TAGS,
    params=PARAMS_PROTOCOLE | {"model": "DummyRegressor", "strategy": "mean", "feature_set": "none", "n_features": 0},
    metrics=cv_dummy,
)

**Observations :**

- La RMSE du `DummyRegressor` (1,695 t/ha) est égale à l'écart-type du rendement.
- Son R² est nul : prédire la moyenne n'explique aucune variation du rendement.

# Régression linéaire — toutes les variables

`Pipeline` scikit-learn : catégorielles → `OneHotEncoder`, numériques conservées telles quelles, puis
`LinearRegression`.

C'est le pipeline complet qui est évalué : dans chaque fold, le preprocessing est réappris sur les seules données
d'apprentissage.

In [9]:
lineaire_toutes = build_pipeline(LinearRegression(), PREDICT_CATEGORICAL, PREDICT_NUMERIC)
cv_lineaire_toutes = cross_validate_regressor(lineaire_toutes, X_train, y_train, cv)

n_colonnes_toutes = count_encoded_columns(PREDICT_CATEGORICAL, PREDICT_NUMERIC, X_train)
print(f"colonnes après encodage : {n_colonnes_toutes}")
print(format_cv_metrics(cv_lineaire_toutes))

colonnes après encodage : 26
RMSE 0.5003 ± 0.0009 t/ha | MAE 0.3993 ± 0.0007 t/ha | R² 0.9129 ± 0.0003


In [10]:
log_model_run(
    RunInfo("linear_regression_all_features_cv", "all_features", TAGS, PARAMS_PROTOCOLE),
    lineaire_toutes, X_train, PREDICT_CATEGORICAL, PREDICT_NUMERIC, cv_lineaire_toutes,
)

**Observations :**

- Les 9 variables deviennent 26 colonnes après encodage.
- RMSE de 0,500 t/ha et R² de 0,913, contre 1,695 t/ha et 0 pour la référence naïve.

# Régression linéaire — variables réduites

Même protocole, sans les variables de `PREDICT_FEATURES_DROP` : `Region`, `Weather_Condition` et `Days_to_Harvest`. Il
reste 6 variables. L'écart avec le modèle à toutes les variables mesure l'apport des variables retirées.

In [11]:
categorielles_reduites, numeriques_reduites = feature_types(VARIABLES_REDUITES, PREDICT_CATEGORICAL, PREDICT_NUMERIC)

lineaire_reduites = build_pipeline(LinearRegression(), categorielles_reduites, numeriques_reduites)
cv_lineaire_reduites = cross_validate_regressor(lineaire_reduites, X_train[VARIABLES_REDUITES], y_train, cv)

n_colonnes_reduites = count_encoded_columns(categorielles_reduites, numeriques_reduites, X_train)
print(f"colonnes après encodage : {n_colonnes_reduites}")
print(format_cv_metrics(cv_lineaire_reduites))

colonnes après encodage : 18
RMSE 0.5003 ± 0.0009 t/ha | MAE 0.3993 ± 0.0007 t/ha | R² 0.9129 ± 0.0003


In [12]:
log_model_run(
    RunInfo("linear_regression_reduced_features_cv", "reduced_features", TAGS, PARAMS_PROTOCOLE),
    lineaire_reduites, X_train[VARIABLES_REDUITES], categorielles_reduites, numeriques_reduites, cv_lineaire_reduites,
)

# Comparaison par validation croisée

Les modèles sont comparés sur la RMSE moyenne de validation croisée.

In [13]:
comparaison = pd.DataFrame(
    {
        "DummyRegressor": cv_dummy,
        "LinearRegression — toutes les variables": cv_lineaire_toutes,
        "LinearRegression — variables réduites": cv_lineaire_reduites,
    }
).T.sort_values("cv_rmse_mean")
comparaison.round(4)

,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_mae_std,cv_r2_mean,cv_r2_std
LinearRegression — variables réduites,0.5003,0.0009,0.3993,0.0007,0.9129,0.0003
LinearRegression — toutes les variables,0.5003,0.0009,0.3993,0.0007,0.9129,0.0003
DummyRegressor,1.6952,0.0027,1.3884,0.0031,-0.0000,0.0000


In [14]:
baisse_rmse = 1 - cv_lineaire_toutes["cv_rmse_mean"] / cv_dummy["cv_rmse_mean"]
ecart_toutes_reduites = cv_lineaire_toutes["cv_rmse_mean"] - cv_lineaire_reduites["cv_rmse_mean"]

print(f"RMSE moyenne : {cv_dummy['cv_rmse_mean']:.3f} → {cv_lineaire_toutes['cv_rmse_mean']:.3f} t/ha, soit {baisse_rmse:.0%} d'erreur en moins")
print(f"écart de RMSE moyenne, toutes − réduites : {ecart_toutes_reduites:+.6f} t/ha")
print(f"écart-type de la RMSE entre folds        : {cv_lineaire_reduites['cv_rmse_std']:.6f} t/ha")

RMSE moyenne : 1.695 → 0.500 t/ha, soit 70% d'erreur en moins
écart de RMSE moyenne, toutes − réduites : +0.000002 t/ha
écart-type de la RMSE entre folds        : 0.000894 t/ha


**Observations :**

- La régression linéaire réduit la RMSE de 70 % par rapport au `DummyRegressor` et explique 91,3 % de la variance du
  rendement.
- Les scores varient très peu d'un fold à l'autre : l'estimation est stable.
- Toutes les variables et les variables réduites donnent des scores quasi identiques.

# Conclusion

**Observations :**

- la régression linéaire fournit une bonne baseline ;
- les 3 variables retirées des variables réduites n'apportent pas de gain visible avec ce modèle ;
- la sélection finale sera confirmée avec les modèles suivants ;
- le test reste réservé à l'évaluation finale.

In [15]:
notify(
    "Agritech /predict",
    f"Baseline terminée — RMSE CV {cv_lineaire_toutes['cv_rmse_mean']:.3f} (toutes les variables), {cv_lineaire_reduites['cv_rmse_mean']:.3f} (variables réduites)",
)